# SECCIÓN 2
## Preparación y validación de los datos

Esta sección continúa el trabajo de exploración inicial. No vuelve a cargar el dataset ni repite los controles generales ya realizados. Su objetivo es revisar la unidad de análisis, justificar las transformaciones necesarias y dejar una tabla con un registro por pedido para el análisis posterior.

## Etapa 0 — Punto de partida y decisiones pendientes

La exploración inicial registró 180.519 filas y 53 columnas, sin filas completamente duplicadas. También identificó valores faltantes en columnas que no son necesarias para el análisis principal, como `Product Description` y `Order Zipcode`.

Antes de comparar ganancias, necesitamos tener en cuenta que la base está organizada por artículos de pedido: un mismo `Order Id` puede aparecer en varias filas. Por eso, la tabla original no debe tratarse directamente como si cada fila fuera un pedido.

La pregunta que viene trabajando el grupo se refiere a las diferencias en la ganancia promedio por pedido según el estado de entrega y a cómo varían por segmento de clientes y región. Para responderla, primero debemos establecer cómo resumir las filas de cada pedido. La interpretación final debe basarse en los resultados del análisis del grupo.

## Etapa 1 — Validar la unidad de análisis y las variables económicas

### 1.1. Comprobar cuántas filas tiene cada pedido

Contamos las filas asociadas a cada `Order Id`. Esto permite confirmar si la unidad de observación original coincide con la unidad que necesita nuestra pregunta de investigación.

In [4]:
filas_por_pedido = df.groupby("Order Id").size()

print(f"Filas de la base original: {len(df):,}")
print(f"Pedidos únicos: {df['Order Id'].nunique():,}")
print(f"Pedidos con una sola fila: {(filas_por_pedido == 1).sum():,}")
print(f"Pedidos con varias filas: {(filas_por_pedido > 1).sum():,}")
print(f"Máximo de filas para un pedido: {filas_por_pedido.max()}")

Filas de la base original: 180,519
Pedidos únicos: 65,752
Pedidos con una sola fila: 19,850
Pedidos con varias filas: 45,902
Máximo de filas para un pedido: 5


### 1.2. Revisar un pedido con varias filas

Observamos un ejemplo con varias filas para comprobar que los registros se distinguen por `Order Item Id` y que los valores económicos pueden cambiar entre artículos. Esto ayuda a evaluar si corresponde resumir las ganancias a nivel de pedido.

In [5]:
id_ejemplo = filas_por_pedido[filas_por_pedido > 1].index[0]

columnas_ejemplo = [
    "Order Id",
    "Order Item Id",
    "Order Item Quantity",
    "Order Item Total",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Benefit per order",
    "Delivery Status"
]

display(
    df.loc[df["Order Id"] == id_ejemplo, columnas_ejemplo]
    .sort_values("Order Item Id")
)

,Order Id,Order Item Id,Order Item Quantity,Order Item Total,Order Item Profit Ratio,Order Profit Per Order,Benefit per order,Delivery Status
77011,2,2,1,193.99,0.47,91.18,91.18,Advance shipping
109322,2,3,5,227.50,0.30,68.25,68.25,Advance shipping
87884,2,4,1,107.89,0.34,36.47,36.47,Advance shipping


### 1.3. Comprobar qué variables se mantienen constantes dentro de cada pedido

El diccionario describe `Benefit per order` como ganancias por pedido. Sin embargo, en la base, `Benefit per order` y `Order Profit Per Order` coinciden fila por fila, pero sus valores pueden variar entre las filas asociadas a un mismo pedido. Por eso, el nombre de las variables por sí solo no alcanza para determinar cómo debe calcularse la ganancia total del pedido.

Comprobamos también que el riesgo, el estado de entrega, el segmento y la región se mantengan constantes dentro de cada pedido. Si alguna variable presenta varios valores para un mismo pedido, no debemos elegir uno arbitrariamente.

In [6]:
variables_constantes = [
    "Late_delivery_risk",
    "Delivery Status",
    "Customer Segment",
    "Order Region"
]

print("¿Coinciden Order Profit Per Order y Benefit per order en todas las filas?")
print(
    df["Order Profit Per Order"].eq(df["Benefit per order"]).all()
)

print("\nCantidad máxima de valores distintos por pedido:")
display(
    df.groupby("Order Id")[variables_constantes]
    .nunique(dropna=False)
    .max()
    .to_frame("Máximo de valores distintos")
)

print("\n¿Order Item Id identifica cada fila?")
print(f"Valores únicos de Order Item Id: {df['Order Item Id'].nunique():,}")
print(f"Filas de la base: {len(df):,}")

¿Coinciden Order Profit Per Order y Benefit per order en todas las filas?
True

Cantidad máxima de valores distintos por pedido:


,Máximo de valores distintos
Late_delivery_risk,1
Delivery Status,1
Customer Segment,1
Order Region,1



¿Order Item Id identifica cada fila?
Valores únicos de Order Item Id: 180,519
Filas de la base: 180,519


### 1.4. Revisar valores negativos y posibles outliers

Antes de construir una tabla por pedido, revisamos los valores negativos de las variables económicas y detectamos posibles valores atípicos mediante el rango intercuartílico (IQR). Estos controles son exploratorios: una ganancia negativa puede representar una pérdida real y un valor extremo no es necesariamente un error. No eliminamos observaciones automáticamente.

También contrastamos `Order Profit Per Order` con el producto `Order Item Total * Order Item Profit Ratio`. Esta comparación permite evaluar coherencia numérica, pero no demuestra por sí sola que esa multiplicación sea la fórmula oficial de la ganancia.

In [7]:
columnas_economicas = [
    "Order Item Total",
    "Order Item Profit Ratio",
    "Order Profit Per Order",
    "Benefit per order",
    "Sales per customer"
]

resumen_negativos = []
for columna in columnas_economicas:
    valores = pd.to_numeric(df[columna], errors="coerce")
    resumen_negativos.append({
        "variable": columna,
        "negativos": int((valores < 0).sum()),
        "porcentaje_negativo": float((valores < 0).mean() * 100),
        "mínimo": valores.min(),
        "máximo": valores.max()
    })

display(pd.DataFrame(resumen_negativos).round(4))

# Contraste numérico exploratorio entre ganancia registrada y total por ratio.
ganancia_calculada = df["Order Item Total"] * df["Order Item Profit Ratio"]
diferencia = (df["Order Profit Per Order"] - ganancia_calculada).abs()
print("Diferencia absoluta entre Order Profit Per Order y Order Item Total * Order Item Profit Ratio:")
display(diferencia.describe().to_frame("diferencia_absoluta").round(4))
print(f"Filas con diferencia absoluta <= 0,02: {(diferencia <= 0.02).sum():,} ({(diferencia <= 0.02).mean()*100:.1f}%)")
print("La relación no es exacta en todas las filas; no se reemplaza la variable original por esta fórmula.")

# Detección exploratoria de outliers mediante IQR.
resumen_outliers = []
for columna in columnas_economicas:
    valores = pd.to_numeric(df[columna], errors="coerce").dropna()
    q1 = valores.quantile(0.25)
    q3 = valores.quantile(0.75)
    iqr = q3 - q1
    inferior = q1 - 1.5 * iqr
    superior = q3 + 1.5 * iqr
    cantidad = int(((valores < inferior) | (valores > superior)).sum())
    resumen_outliers.append({
        "variable": columna,
        "límite_inferior_IQR": inferior,
        "límite_superior_IQR": superior,
        "posibles_outliers": cantidad,
        "porcentaje": cantidad / len(valores) * 100
    })

display(pd.DataFrame(resumen_outliers).round(4))

,variable,negativos,porcentaje_negativo,mínimo,máximo
0,Order Item Total,0,0.00,7.49,"1,939.99"
1,Order Item Profit Ratio,33784,18.71,-2.75,0.50
2,Order Profit Per Order,33784,18.71,"-4,274.98",911.80
3,Benefit per order,33784,18.71,"-4,274.98",911.80
4,Sales per customer,0,0.00,7.49,"1,939.99"


Diferencia absoluta entre Order Profit Per Order y Order Item Total * Order Item Profit Ratio:


,diferencia_absoluta
count,"180,519.00"
mean,0.22
std,0.41
min,0.00
25%,0.00
50%,0.00
75%,0.33
max,9.30


Filas con diferencia absoluta <= 0,02: 115,474 (64.0%)
La relación no es exacta en todas las filas; no se reemplaza la variable original por esta fórmula.


,variable,límite_inferior_IQR,límite_superior_IQR,posibles_outliers,porcentaje
0,Order Item Total,-110.15,461.93,1943,1.08
1,Order Item Profit Ratio,-0.34,0.78,17300,9.58
2,Order Profit Per Order,-79.70,151.50,18942,10.49
3,Benefit per order,-79.70,151.50,18942,10.49
4,Sales per customer,-110.15,461.93,1943,1.08


**Decisión metodológica:** no se eliminan ganancias negativas ni posibles outliers por el solo hecho de ser negativos o extremos. La variable `Order Profit Per Order` se agrega por `Order Id` como suma de los importes registrados en las filas del pedido, interpretándola como una medida derivada de los registros de artículos. Esta decisión es compatible con que los importes varíen entre filas, pero debe leerse con la limitación documentada: el diccionario no confirma una fórmula de agregación y la relación con `Order Item Total * Order Item Profit Ratio` no es exacta en todas las filas. La columna `margen_ganancia_pedido` se considera una medida derivada exploratoria, no una variable original del dataset.